# Practice Lab: Document Processing Pipeline (Lesson 8.2)

Module 8 · 8 exercises · Loaders + chunking + metadata + OCR + India processing

Exercises 1, 2, 3, 5, 7, 8 run locally (pure Python + numpy).


# Lesson 8.2: Document Processing Pipeline

8 exercises: 2E + 3M + 3C

Exercises 1, 2, 3, 5, 7, 8 run **locally** (pure Python). Ex 4, 6 are architecture/design.


In [ ]:
import re, hashlib, unicodedata
import numpy as np


---
## Exercise 1 (Easy): PDF → Markdown → Chunks


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import re

markdown_doc = """# Netsetos GenAI Course

## About
Netsetos is an educational technology company founded in 2024 in Hyderabad India. We specialize in Generative AI Data Science and Cloud Computing courses for Indian software engineers. The flagship GenAI Complete Course covers 14 modules across 146 hours.

## Pricing
The GenAI course costs 14999 rupees one-time payment. Includes lifetime access Discord weekly live sessions certificate and 5000 GCP credits. EMI via Razorpay starting at 2500 rupees per month.

## Refund Policy
Full refund within 7 days of purchase. 50 percent refund from day 8 to 30. No refunds after 30 days. Processed within 5 business days.

## Live Classes
Live classes daily at 7 PM IST on YouTube. Recordings in 2 hours. 85 percent completion rate 92 percent placement rate 4.8 star rating.

## Technical Requirements
Laptop with 8GB RAM and stable internet 10 Mbps minimum. Google account for GCP access. No prior AI experience required."""

def chunk_recursive(text, max_size=100):
    seps = ["\n\n", "\n", ". ", " "]
    chunks = []
    def _split(t, si=0):
        if len(t.split()) <= max_size:
            if len(t.split()) >= 15: chunks.append(t.strip())
            return
        if si >= len(seps):
            chunks.append(" ".join(t.split()[:max_size]))
            return
        parts = t.split(seps[si])
        cur = ""
        for p in parts:
            if len((cur + seps[si] + p).split()) <= max_size:
                cur += seps[si] + p
            else:
                if cur.strip(): _split(cur.strip(), si + 1)
                cur = p
        if cur.strip(): _split(cur.strip(), si + 1)
    _split(text)
    return chunks

clean = re.sub(r'#{1,6}\s+', '', markdown_doc)
clean = re.sub(r'[*_`~]+', '', clean)

print(f"Document: {len(clean.split())} words\n")
print(f"{'Size':<10} {'Chunks':<8} {'Avg':<8} {'Min':<8} {'Max'}")
print("=" * 42)
for size in [64, 128, 256, 512]:
    ch = chunk_recursive(clean, size)
    if ch:
        avg = sum(len(c.split()) for c in ch) // len(ch)
        print(f"{size:<10} {len(ch):<8} {avg:<8} "
              f"{min(len(c.split()) for c in ch):<8} "
              f"{max(len(c.split()) for c in ch)}")

print(f"\nChunks at 128w:")
for i, c in enumerate(chunk_recursive(clean, 128)):
    sec = "Unknown"
    for kw, s in [("refund","Refund"),("rupees","Pricing"),("live","Classes"),("netsetos","About"),("8gb","Technical")]:
        if kw in c.lower(): sec = s; break
    print(f"  [{i}] {sec}: {c[:60]}...")


</details>


---
## Exercise 2 (Easy): Multi-Format Loader


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import re

class DocLoader:
    FORMATS = {".pdf": "pymupdf4llm", ".html": "Trafilatura", ".docx": "mammoth",
               ".csv": "pandas to_markdown", ".xlsx": "pandas to_markdown",
               ".txt": "direct read", ".md": "strip formatting",
               ".pptx": "python-pptx", ".eml": "email stdlib"}
    def load(self, name, content):
        ext = "." + name.rsplit(".",1)[-1].lower()
        text = re.sub(r'\s+', ' ', content).strip()
        return {"source": name, "format": ext, "tool": self.FORMATS.get(ext, "unknown"),
                "text": text, "words": len(text.split())}

loader = DocLoader()
files = [
    ("faq.pdf", "Netsetos offers GenAI courses in Hyderabad with 14 modules"),
    ("refund.html", "<p>Full refund within 7 days. 50% from 8-30 days.</p>"),
    ("pricing.docx", "GenAI course Rs 14999 with lifetime access and certificate"),
    ("students.csv", "name,course,batch\nRahul,GenAI,2026-Q1"),
    ("schedule.txt", "Live classes daily 7 PM IST on YouTube"),
]
print(f"{'File':<20} {'Format':<8} {'Words':<8} {'Tool'}")
print("=" * 55)
for name, content in files:
    r = loader.load(name, content)
    print(f"  {r['source']:<18} {r['format']:<8} {r['words']:<8} {r['tool']}")

print(f"\nSupported: {len(loader.FORMATS)} formats")
print(f"Pattern: ANY format -> Markdown -> Chunk")


</details>


---
## Exercise 3 (Medium): Chunking Strategy Benchmark


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import re, hashlib
import numpy as np

def fake_embed(text, dim=64):
    h = hashlib.sha256(text.lower().encode()).digest()
    vec = np.array([b/255.0 for b in h[:dim]])
    kw = {"refund":0,"price":1,"cost":2,"rupees":3,"live":4,"class":5,"course":6,
          "completion":8,"placement":9,"emi":10,"certificate":11,"youtube":13,"ram":15}
    for w, i in kw.items():
        if w in text.lower().split() and i < dim: vec[i] += 0.5
    return vec / np.linalg.norm(vec)

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

corpus = ("Netsetos is an edtech company in Hyderabad India offering GenAI courses.\n\n"
          "GenAI course costs 14999 rupees one-time. Includes lifetime access Discord "
          "live sessions certificate and 5000 GCP credits. EMI via Razorpay at 2500 per month.\n\n"
          "Full refund within 7 days. 50 percent from 8-30 days. No refunds after 30 days.\n\n"
          "Live classes daily 7 PM IST on YouTube. Recordings in 2 hours. "
          "85 percent completion 92 percent placement 4.8 star rating.\n\n"
          "Laptop with 8GB RAM stable internet 10 Mbps. Google account for GCP. No prior AI needed.")

def chunk_fixed(t, sz=60, ov=10):
    w = t.split(); ch = []
    for i in range(0, len(w), sz-ov):
        c = " ".join(w[i:i+sz])
        if len(c.split()) >= 15: ch.append(c)
    return ch

def chunk_recursive(t, ms=60):
    seps = ["\n\n","\n",". "," "]; ch = []
    def _s(t, si=0):
        if len(t.split()) <= ms:
            if len(t.split()) >= 15: ch.append(t.strip())
            return
        if si >= len(seps): ch.append(" ".join(t.split()[:ms])); return
        parts = t.split(seps[si]); cur = ""
        for p in parts:
            if len((cur+seps[si]+p).split()) <= ms: cur += seps[si]+p
            else:
                if cur.strip(): _s(cur.strip(), si+1)
                cur = p
        if cur.strip(): _s(cur.strip(), si+1)
    _s(t); return ch

def chunk_semantic(t, th=0.2):
    sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+', t) if len(s.split()) >= 3]
    if not sents: return [t]
    ch, cur = [], [sents[0]]
    for i in range(1, len(sents)):
        prev = set(" ".join(cur[-2:]).lower().split())
        curr = set(sents[i].lower().split())
        if len(prev & curr)/max(len(prev|curr),1) >= th: cur.append(sents[i])
        else: ch.append(" ".join(cur)); cur = [sents[i]]
    if cur: ch.append(" ".join(cur))
    return [c for c in ch if len(c.split()) >= 10]

queries = [("Refund policy?","refund"),("Course cost?","14999"),("Live classes?","7 PM"),
           ("Completion rate?","85"),("Technical requirements?","8gb"),("EMI?","emi"),
           ("Youtube?","youtube"),("Certificate?","certificate"),("Location?","hyderabad"),("GCP?","gcp")]

print(f"Chunking Benchmark ({len(corpus.split())}w, {len(queries)} queries):\n")
results = {}
for name, fn in [("Fixed(60)", lambda: chunk_fixed(corpus)),
                  ("Recursive(60)", lambda: chunk_recursive(corpus)),
                  ("Semantic", lambda: chunk_semantic(corpus))]:
    ch = fn(); embs = [fake_embed(c) for c in ch]
    hits = 0
    for q, exp in queries:
        qe = fake_embed(q)
        top3 = sorted([(i, cosine(qe, e)) for i, e in enumerate(embs)], key=lambda x: -x[1])[:3]
        if exp.lower() in " ".join(ch[i] for i, _ in top3).lower(): hits += 1
    r = hits/len(queries); results[name] = r
    print(f"  {name:<18} {len(ch):>2} chunks  recall@3: {r:.0%} {'█'*int(r*20)}")

print(f"\nWinner: {max(results, key=results.get)}")
print(f"Recursive: respects paragraphs, Vecta benchmark #1 (69%)")


</details>


---
## Exercise 5 (Medium): Metadata Enrichment Pipeline


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import hashlib
import numpy as np

def fake_embed(t, dim=64):
    h = hashlib.sha256(t.lower().encode()).digest()
    vec = np.array([b/255.0 for b in h[:dim]])
    kw = {"refund":0,"price":1,"policy":3,"2025":4,"2023":5}
    for w, i in kw.items():
        if w in t.lower() and i < dim: vec[i] += 0.5
    return vec / np.linalg.norm(vec)

def cosine(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

chunks = [
    {"text": "Refund policy 2025: Full refund 7 days. 50% from 8-30 days.",
     "source": "refund_2025.pdf", "date": "2025-06"},
    {"text": "Refund policy 2023: Full refund 14 days. All sales final after.",
     "source": "refund_2023.pdf", "date": "2023-01"},
    {"text": "Course pricing Rs 14999 lifetime access EMI Razorpay.",
     "source": "pricing_2025.pdf", "date": "2025-06"},
    {"text": "Live classes 7 PM IST YouTube. 85% completion rate.",
     "source": "faq_2025.pdf", "date": "2025-06"},
]

for c in chunks:
    c["emb"] = fake_embed(c["text"])
    c["hash"] = hashlib.md5(c["text"].encode()).hexdigest()[:8]

q = "What is the current refund policy?"
qe = fake_embed(q)

print("Without Filter:")
scores = sorted([(i, cosine(qe, c["emb"])) for i, c in enumerate(chunks)], key=lambda x: -x[1])
for i, s in scores[:3]:
    print(f"  [{s:.4f}] {chunks[i]['source']} ({chunks[i]['date']})")

print(f"\nWith Filter (date >= 2025):")
filtered = [(i, c) for i, c in enumerate(chunks) if c["date"] >= "2025"]
fscores = sorted([(i, cosine(qe, c["emb"])) for i, c in filtered], key=lambda x: -x[1])
for i, s in fscores[:3]:
    print(f"  [{s:.4f}] {chunks[i]['source']} ({chunks[i]['date']})")

hashes = [c["hash"] for c in chunks]
print(f"\nDedup: {len(chunks)} chunks, {len(set(hashes))} unique")
print(f"Impact: date filter returns correct 2025 policy")
print(f"MetaRAG: +9% precision from LLM-generated metadata")


</details>


---
## Exercise 7 (Challenge): Production Ingestion Pipeline


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import hashlib

class IncrementalIndex:
    def __init__(self):
        self.seen = {}
    def should_process(self, doc_id, content):
        h = hashlib.sha256(content.encode()).hexdigest()[:16]
        if doc_id in self.seen:
            if self.seen[doc_id] == h: return False, "unchanged"
            self.seen[doc_id] = h; return True, "modified"
        self.seen[doc_id] = h; return True, "new"

print("Production Ingestion Architecture:")
for step in ["1. File Watcher (watchdog/S3)", "2. Classifier (PDF/DOCX/HTML)",
             "3. Parser (PyMuPDF4LLM -> Docling -> VLM)", "4. Chunker (recursive 512)",
             "5. Embedder (async, batched)", "6. Upsert (hash dedup)", "7. Dead Letter Queue"]:
    print(f"  {step}")

print(f"\nIncremental Indexing:")
idx = IncrementalIndex()
for doc_id, content, desc in [
    ("doc_1", "Original content", "First upload"),
    ("doc_1", "Original content", "Same content"),
    ("doc_1", "Updated content", "Modified"),
    ("doc_2", "New document", "New file"),
]:
    ok, status = idx.should_process(doc_id, content)
    print(f"  [{'PROCESS' if ok else 'SKIP':>7}] {doc_id}: {desc} -> {status}")

print(f"\nMonitoring: docs/hr, failure rate, processing time, store growth")
print(f"Optimization: async embed cut 30min -> 2.5min (92%)")


</details>


---
## Exercise 8 (Challenge): Hindi Document Processing


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import re, unicodedata

def classify_hindi_pdf(text):
    dev = len(re.findall(r'[\u0900-\u097F]', text))
    asc = len(re.findall(r'[a-zA-Z]', text))
    total = len(text.strip())
    if total < 10: return "scanned", "Sarvam Vision OCR"
    if dev > total * 0.3: return "unicode_hindi", "pymupdf4llm"
    if asc > total * 0.5 and dev == 0:
        if any(ord(c) > 127 and ord(c) < 256 for c in text[:100]):
            return "legacy_font", "Font2Unicode first"
        return "english", "pymupdf4llm"
    return "mixed", "Language detect per para"

tests = [
    ("\u092d\u093e\u0930\u0924 \u0938\u0930\u0915\u093e\u0930 \u0915\u093e \u0906\u0926\u0947\u0936", "Gov Hindi"),
    ("Supreme Court of India orders", "English legal"),
    ("", "Scanned doc"),
    ("\u092e\u0936\u0940\u0928 learning \u0914\u0930 AI", "Mixed"),
]
print("Hindi PDF Classification:")
for text, desc in tests:
    t, a = classify_hindi_pdf(text)
    print(f"  [{t:<14}] {desc}: {a}")

# Normalization check
print(f"\nDevanagari Normalization:")
for sample in ["\u0915\u093c", "\u0958", "\u0915\u094d\u0937"]:
    nfc = unicodedata.normalize('NFC', sample)
    print(f"  '{sample}' (U+{ord(sample[0]):04X}): NFC changed={sample != nfc}")

print(f"\nFont2Unicode: kru2uni (IIIT Hyderabad) for Kruti Dev")
print(f"Mixed flow: FastText detect -> Sarvam (Hindi) / standard (English)")
print(f"Embed: BGE-M3 (100+ langs), store language tag as metadata")
print(f"180+ non-standard Devanagari fonts exist in Indian PDFs")


</details>


---
## Exercise 4 (Medium): Table Extraction Comparison
Architecture/design. See practice-lab-8_2.html.


In [ ]:
# Exercise 4: Table Extraction Comparison
print("Table Extraction Comparison:")
print("=" * 55)

# Sample table data
table_data = [
    {"plan": "Basic", "price": "₹4,999", "modules": 5, "support": "Email"},
    {"plan": "Pro", "price": "₹14,999", "modules": 14, "support": "Discord + Live"},
    {"plan": "Enterprise", "price": "₹49,999", "modules": 14, "support": "1-on-1 Mentor"},
]

# Format 1: Markdown table (pymupdf4llm output)
print("1. Markdown Table (pymupdf4llm):")
print("| Plan | Price | Modules | Support |")
print("|------|-------|---------|---------|")
for r in table_data:
    print(f"| {r['plan']:<10} | {r['price']:<9} | "
          f"{r['modules']:<7} | {r['support']:<14} |")

# Format 2: CSV (pdfplumber output)
print(f"\n2. CSV (pdfplumber):")
print("Plan,Price,Modules,Support")
for r in table_data:
    print(f"{r['plan']},{r['price']},{r['modules']},"
          f"{r['support']}")

# Format 3: Natural language (best for retrieval)
print(f"\n3. Natural Language (best for RAG retrieval):")
nl_descriptions = []
for r in table_data:
    desc = (f"The {r['plan']} plan costs {r['price']} "
            f"and includes {r['modules']} modules with "
            f"{r['support']} support.")
    nl_descriptions.append(desc)
    print(f"  {desc}")

# Parser comparison
print(f"\nPDF Parser Table Accuracy:")
parsers = [
    ("pymupdf4llm", "Markdown pipes", "0.12s", "Good (born-digital)"),
    ("pdfplumber", "Row/col extraction", "0.10s", "96% (visible grids)"),
    ("Docling (IBM)", "TableFormer ML", "0.5-2s", "97.9% (any layout)"),
    ("VLMs (GPT-4o)", "Vision understanding", "2-10s", "Best (any format)"),
]
print(f"{'Parser':<16} {'Method':<20} {'Speed':<8} {'Accuracy'}")
print("-" * 60)
for name, method, speed, acc in parsers:
    print(f"  {name:<14} {method:<20} {speed:<8} {acc}")

print(f"\nKey Rules:")
print(f"  1. NEVER chunk across table rows")
print(f"  2. Keep entire table as one chunk")
print(f"  3. Convert to natural language for better retrieval")
print(f"  4. Store original format as metadata for display")

---
## Exercise 6 (Challenge): OCR Preprocessing Pipeline
Architecture/design. See practice-lab-8_2.html.


In [ ]:
# Exercise 6: OCR Preprocessing Pipeline
print("OCR Preprocessing Pipeline:")
print("=" * 55)

steps = [
    ("1. Rescale", "300 DPI minimum (upscale if lower)",
     "cv2.resize with INTER_CUBIC"),
    ("2. Grayscale", "Remove color channel noise",
     "cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)"),
    ("3. Denoise", "Remove speckle noise",
     "cv2.medianBlur(gray, 3)"),
    ("4. Binarize", "Black text on white background",
     "cv2.threshold (Otsu's method)"),
    ("5. Deskew", "Correct rotation/skew",
     "minAreaRect angle correction"),
]

for name, purpose, code in steps:
    print(f"  {name}")
    print(f"    Purpose: {purpose}")
    print(f"    Code: {code}")

# OCR engine comparison
print(f"\nOCR Engine Comparison:")
engines = [
    ("Tesseract v5", "100+", "CPU", "Free", "Baseline"),
    ("PaddleOCR v5", "109", "CPU/GPU", "Free", "+13% over v4"),
    ("Surya OCR", "90+", "GPU", "Free (GPL)", "PDF-native"),
    ("Azure Doc Intel", "300+", "Cloud", "$1.50/1K", "99%+ accuracy"),
    ("Sarvam Vision", "23 Indian", "Cloud", "Rs 1.50/pg", "87.4% WA Indic"),
]
print(f"{'Engine':<18} {'Langs':<10} {'Compute':<10} "
      f"{'Cost':<12} {'Notes'}")
print("-" * 65)
for name, langs, compute, cost, notes in engines:
    print(f"  {name:<16} {langs:<10} {compute:<10} "
          f"{cost:<12} {notes}")

# Cost calculation for 10K pages
print(f"\nCost for 10,000 Pages:")
costs = [
    ("Tesseract/PaddleOCR", "Free (self-host)", "Rs 0"),
    ("Azure Doc Intel", "$15.00", "Rs 1,260"),
    ("Sarvam Vision", "Rs 15,000", "Rs 15,000"),
    ("GPT-4o Vision", "$100-500", "Rs 8,400-42,000"),
]
for engine, usd, inr in costs:
    print(f"  {engine:<22} {usd:<16} {inr}")

print(f"\nRecommendation:")
print(f"  English: PaddleOCR v5 (free, 109 langs)")
print(f"  Indian: Sarvam Vision (87.4% WA, 22 langs)")
print(f"  Enterprise: Azure Doc Intel (99%+, 300+ langs)")
print(f"  Preprocessing adds 20-40% accuracy improvement")